In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
import umap
import hdbscan
from scipy.stats import chi2_contingency

import src.fct_data
import importlib
importlib.reload(src.fct_data) 

from src.fct_data import raw_data_cleaning


/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Extraction des données



Traitement pour les 3 années disponibles sur le `gitlab` 

In [2]:
#on récupère un df avec métadonnées + texte nettoyé des stopwords
meta_et_texts=raw_data_cleaning("data/archelect_search.zip","data/text_files")

/home/onyxia/work/NLP_3A/src/fct_data.py:12: DtypeWarning: Columns (8,9,10,12,28,29,30,31,32,33,34,35,36,37,38,39,40,41) have mixed types. Specify dtype option on import or set low_memory=False.
  metadonnees = pd.read_csv("data/archelect_search.zip", compression="zip")


Traitement de data/text_files/1988/legislatives.zip
Traitement de data/text_files/1981/legislatives.zip
Traitement de data/text_files/1993/legislatives.zip
Traitement de data/text_files/1993/presidentielle.zip
Nombre total de documents extraits : 12746


[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## [NEW] Identifcation des thèmes pour l'élection 1981

### Première approche : BoW + IDF 

Pour montrer que ça ne marche pas bien pcq on rate les thèmes

In [3]:
#1981
texts_1981 = meta_et_texts[meta_et_texts['annee'] == 1981]
print(f"Nombre de documents 1981 : {len(texts_1981)}")

#on selectionne les partis présentants le plus de candidats
parti_counts = texts_1981['suppleant-soutien'].value_counts()
parti_counts = parti_counts.drop("non mentionné", errors='ignore')
top_partis = parti_counts.head(6).index.tolist()

rows=[]

#matrice BoW IDF
vectorizer = TfidfVectorizer(
    stop_words=None,   
    max_features=30    # garder les 30 mots les plus fréquents
)

for parti in top_partis:
    # Filtrer les textes de ce parti
    texts_parti = texts_1981[texts_1981['suppleant-soutien'] == parti]['texte_clean']
    
    idf_matrix = vectorizer.fit_transform(texts_parti)

    idf_df = pd.DataFrame(idf_matrix.toarray(), columns=vectorizer.get_feature_names_out())
    top_parti = idf_df.mean(axis=0).sort_values(ascending=False).index.tolist()
      # Ajouter au tableau
    rows.append({
        "parti": parti,
        "top_words": top_parti
    })


top_words_df = pd.DataFrame(rows)
top_words_df['top_words'] = top_words_df['top_words'].apply(lambda x: ", ".join(x))
top_words_df

Nombre de documents 1981 : 3121


,parti,top_words
0,Parti socialiste,"france, socialiste, majorité, plus, politique,..."
1,Parti communiste français,"gauche, majorité, changement, communistes, com..."
2,Lutte ouvrière,"gauche, mitterrand, faut, travailleurs, plus, ..."
3,Parti socialiste unifié,"psu, majorité, gauche, cest, plus, politique, ..."
4,Rassemblement pour la République,"france, plus, politique, fonds, po, majorité, ..."
5,Rassemblement pour la République;Union pour la...,"plus, majorité, france, juin, candidat, politi..."


## Représentation textuelle

Prend environ 2min grâce à la parallélisation

In [ ]:
import time
import numpy as np
from joblib import Parallel, delayed
import math

# -----------------------------
# 1️⃣ Charger le modèle CamemBERT
# -----------------------------
device = 'cpu'  # ou 'cuda' si GPU disponible
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device=device)

# -----------------------------
# 2️⃣ Paramètres
# -----------------------------
texts = meta_et_texts['texte_clean'].tolist()  # liste des textes
chunk_size = 1000       # taille des chunks pour parallélisation
batch_size = 2048       # batch_size à l'intérieur du modèle
num_cores = 48          # nombre de threads CPU à utiliser pour joblib

# découper en chunks
chunks = [texts[i:i+chunk_size] for i in range(0, len(texts), chunk_size)]
num_chunks = len(chunks)

print(f"Nombre total de chunks : {num_chunks}")

# -----------------------------
# 3️⃣ Fonction pour encoder un chunk
# -----------------------------
def encode_chunk(chunk, chunk_idx):
    start = time.time()
    embeddings = model.encode(chunk, batch_size=batch_size, convert_to_numpy=True)
    end = time.time()
    elapsed = end - start
    print(f"✅ Chunk {chunk_idx+1}/{num_chunks} traité en {elapsed:.2f}s")
    return embeddings, elapsed

# -----------------------------
# 4️⃣ Encodage parallèle avec suivi du temps
# -----------------------------
start_total = time.time()

results = Parallel(n_jobs=num_cores)(
    delayed(encode_chunk)(chunk, idx) for idx, chunk in enumerate(chunks)
)

# Séparer embeddings et temps par chunk
embeddings_list = [res[0] for res in results]
chunk_times = [res[1] for res in results]

# Concaténer tous les embeddings
embeddings = np.vstack(embeddings_list)
total_time = time.time() - start_total

# -----------------------------
# 5️⃣ Affichage final
# -----------------------------
print(f"\n✅ Embeddings générés pour {len(texts)} textes !")
print(f"Shape des embeddings : {embeddings.shape}")
print(f"Temps total écoulé : {total_time/60:.2f} minutes")
print(f"Temps moyen par chunk : {np.mean(chunk_times):.2f} secondes")
print(f"Temps estimé si encore {num_chunks} chunks : {np.mean(chunk_times)*(num_chunks-1)/60:.2f} minutes")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 753.35it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Nombre total de chunks : 13
✅ Chunk 1/13 traité en 57.37s
✅ Chunk 3/13 traité en 53.31s
✅ Chunk 2/13 traité en 57.74s
✅ Chunk 4/13 traité en 55.29s
✅ Chunk 5/13 traité en 54.86s
✅ Chunk 13/13 traité en 36.83s
✅ Chunk 6/13 traité en 54.07s
✅ Chunk 8/13 traité en 50.15s
✅ Chunk 7/13 traité en 55.66s
✅ Chunk 10/13 traité en 49.49s
✅ Chunk 9/13 traité en 54.44s
✅ Chunk 11/13 traité en 49.07s
✅ Chunk 12/13 traité en 50.30s

✅ Embeddings générés pour 12746 textes !
Shape des embeddings : (12746, 384)
Temps total écoulé : 1.48 minutes
Temps moyen par chunk : 52.20 secondes
Temps estimé si encore 13 chunks : 10.44 minutes


## Classification par thème

Ci-dessous, on transforme notre embedding en vecteurs de bcp plus petite dimension
Prend moins d'une minute

In [7]:
umap_embeddings = umap.UMAP(
    n_neighbors=15, min_dist=0.0, n_components=5, random_state=42
).fit_transform(embeddings)
print(f"UMAP embeddings shape: {umap_embeddings.shape}")

/opt/python/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP embeddings shape: (12746, 5)


DBScan ci-dessous va créer des clusters thématiques

In [8]:
print("Clustering avec HDBSCAN...")
clusterer = hdbscan.HDBSCAN(min_cluster_size=10)
meta_et_texts['theme_cluster'] = clusterer.fit_predict(umap_embeddings)
print(f"Clusters trouvés : {meta_et_texts['theme_cluster'].nunique()} (le -1 correspond aux outliers)")


Clustering avec HDBSCAN...
Clusters trouvés : 132 (le -1 correspond aux outliers)


## Croisement avec les professions des candidats

On fait un peu de visualisation

In [12]:
plt.figure(figsize=(12,6))
sns.countplot(x='theme_cluster', hue='profession', data=meta_et_texts)
plt.title("Répartition des thèmes abordés par métier")
plt.xlabel("Thème")
plt.ylabel("Nombre de professions de foi")
plt.legend(title="Métier")
plt.show()

ValueError: Could not interpret value `profession` for `hue`. An entry with this name does not appear in `data`.

<Figure size 1200x600 with 0 Axes>

## Analyses statistiques diverses

In [ ]:
contingency_table = pd.crosstab(meta_et_texts['profession'], meta_et_texts['theme_cluster'])
chi2, p, dof, expected = chi2_contingency(contingency_table)
print(f"Chi2 = {chi2:.2f}, p-value = {p:.4f}")